# Brain-state transitions vs. anxiety and depression

Figures on Cam-CAN HADS (anxiety and depression categories):

1. **Four rankings side by side.** `bstm_selection` (transition matrices) and
   `fcm_selection` (static connectivity) at rest and during the movie, with **every row**
   of their `summary.csv`: every model, arm, atlas, window and state set (`FIG1_MODELS`,
   `FIG1_ARMS` narrow it). One block of panels per model, because every row already carries
   the covariates and so stands on its own model's baseline. The dashed line is that model's
   covariates-only baseline (Age + Sex, fitted here), with a ± sd band. Large markers are each
   tree's best row at rest (●) and in the movie (◆): filled where it won, hollow where it is
   carried over. Coloured rows, ± sd over seeds, are above the baseline + sd at rest **and** in
   the movie: the best `HIGHLIGHT_MAX` of them are named, and the table lists all.

   **1.2–1.6: what drives the score.** The same rows, grouped by one axis of the grid at a
   time: window size, atlas, model, state method, arm (`FACTORS`).
2. **Volcano, bstm only.** For rest and movie, the `TOP_N` best state sets of the `MODEL` /
   `ARM` ranking. Every transition cell is tested against HADS with an effect size on x
   (`X_AXIS`), adjusted for `COVARIATES`. Colour shows the state set, and the legend rank (#1…)
   is the set's position in that panel's ranking.

   **2.1: every variable of an arm.** The same 2 × 2 over the occupancy and dynamics stored
   beside the transitions, not only the cells (`ARM_21`, e.g. `dynamics` or `all [cond]`).
3. **The multiverse.** Every variable of every state set (each atlas × window × state
   definition), with no ranking or selection. One dot is one variable in one set. The question
   is whether an association holds across the analysis choices or only in one corner of the grid.

The selection runs (`fmri-decomp select-bstm` / `select-fcm`) produce the rankings.
This notebook only reads them, plus the `transitions/` tables, and writes nothing
to `outputs/`. Everything you might want to change is in the next cell.

In [ ]:
import os
from pathlib import Path

ROOT = Path(os.environ.get("FMRIDECOMP_OUTPUTS",
                           "/project/6008063/tamires/DecomposingfMRI/outputs"))
CAMCAN = Path(os.environ.get("CAMCAN_PHENO",
                             "/project/6008063/tamires/cohorts/camcan/dataman/useraccess/"
                             "opendata/paule_toussaint_camcan01870"))
PHENO_FILES = [(CAMCAN / "approved_data.tsv", "\t"),   # HADS
               (CAMCAN / "standard_data.csv", ",")]     # Age, Sex
PHENO_ID = "CCID"

# label -> column. Labels are what the figures say.
TARGETS = {"anxiety":    "additional_HADS_anx_category",
           "depression": "additional_HADS_dep_category"}
ORDINAL_LEVELS = ["Normal", "Mild", "Moderate", "Severe"]   # low -> high

# task label in the *_selection trees -> task value inside the transition tables
TASKS = {"rest": "rest", "movie": "movie"}                  # matched case-insensitively
COHORTS = ["camcan", "camcan_rest"]                         # cohort= dirs to read

MODEL = "ridge"                                  # figure 2 picks its sets from this model's ranking
ARM = {"bstm": "cells [joint]", "fcm": "edges"}  # ... and this arm (figure 1 shows every arm)

TOP_N = 10                                       # state sets per volcano panel
COVARIATES = ["Age", "Sex", "n_transitions"]
CATEGORICAL = ["Sex"]
ALPHA = 0.10                                     # BH-FDR, per panel
SAVE_DIR = None                                  # e.g. Path("figures/08") to save PNG + SVG

# Figure 1 shows every row of the four summary.csv files, narrowed by these.
FIG1_MODELS = None                               # None = all but logistic (AUC, another scale)
FIG1_ARMS = None                                 # None = every arm, e.g. ["cells [joint]", "edges"]

# The dashed lines: one covariates-only model per (phenotype, model), fitted HERE
# on BASELINE_COVARIATES with the pipeline's own CV (score_once), on everyone the
# bstm runs scored. The same lines in all four columns.
BASELINE_COVARIATES = ["Age", "Sex"]
BASELINE_ARM = "(covariates only)"               # how the runs label their own baseline rows
SEEDS = [0, 1, 2, 3, 4]                          # as select-bstm / select-fcm
MARGIN_SD = 0                                    # "above" = mean - MARGIN_SD * row sd > baseline + its sd
HIGHLIGHT_MAX = 10                               # rows named per panel (best worst-task margin); the table has all
REPO = Path.cwd().resolve().parent               # for fmri_decomposition.score_once
N_JOBS = (len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity")
          else os.cpu_count())                   # cores for the baseline fits (your SLURM share)

# Figures 1.2-1.6: one per axis of the grid. column -> title. Also available: "K", "embedding".
FACTORS = {"window": "window size", "atlas": "atlas", "model": "model",
           "method": "state method", "arm": "arm"}
FACTOR_TREES = ["bstm"]                          # add "fcm" to pool its rows where the axis exists

# Figure 2's x axis. All three adjust for COVARIATES by regression.
#   "log2fc"  log2( mean cell, CASE_FROM and above / mean cell, Normal ), with a pseudocount
#   "slope"   change in the cell's probability per HADS level (raw units)
#   "r"       partial Spearman r (p is then a function of |r|: the narrow V)
X_AXIS = "log2fc"
CASE_FROM = "Mild"                               # log2fc: this level and above = elevated
SHRINK = 1.0                                     # log2fc pseudocount, in units of the average cell:
                                                 #   0 = raw ratio (widest), 1 = moderate, 5+ = narrow
MIN_PRESENT = 0.10                               # drop cells non-zero in < this share of subjects
XLIM = {"log2fc": 2, "slope": None, "r": 1}[X_AXIS]   # same x range ±XLIM in every volcano panel; None = fit each
FC_LINE = {"log2fc": 0.25, "slope": None, "r": 0.1}[X_AXIS]   # vertical dashed lines at ±this; None = off
LABEL_TOP = 3                                    # names shown when nothing survives FDR
LABEL_MAX = 25                                   # most names per panel (every FDR survivor, up to this)

# Figure 2.1: the arm whose variables are tested AND whose ranking picks the sets.
#   cells [joint] | cells [cond] | occupancy | dynamics | summary | all [joint] | all [cond]
ARM_21 = "all [cond]"                            # e.g. "dynamics"
RANK_21 = None                                   # whose ranking picks 2.1's sets: None = ARM_21's own;
                                                 # ARM["bstm"] = figure 2's sets, so with ARM_21 = "all [joint]"
                                                 # figure 2's dots are an exact subset of 2.1's

# Figure 3, the multiverse: every variable of ARM_3 in EVERY state set, no ranking.
ARM_3 = "all [cond]"
COLOR_BY_3 = ["block", "window", "atlas", "method", "K"]   # one figure per colouring

# Figure 4: the volcano drawn on one state set's state graph. None = the set with the best
# MODEL / ARM score averaged over the four panels; or e.g.
# SET_4 = dict(atlas="yeo7", window_s=30, states="HMM1_pca3_8")
SET_4 = None
EDGE_TOP = 12                                    # the Normal group's strongest arrows (+ every red one)


In [ ]:
import sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy import stats

INK, INK2, MUTED, SURFACE = "#0b0b0b", "#52514e", "#a3a29d", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK2, "axes.titlesize": 11,
    "axes.titlecolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "font.size": 9, "legend.frameon": False})

def hive(path, key):
    """One `key=value` partition from a path, by name."""
    return next((p.split("=", 1)[1] for p in Path(path).parts if p.startswith(f"{key}=")), None)

def fmt_window(w):
    return str(int(w)) if float(w).is_integer() else str(w)

def set_label(r):
    """A readable name for one summary row: state set or static-FC atlas."""
    if r.get("tree") == "fcm":
        return r["atlas"]
    w = "1 TR" if float(r["window_s"]) < 0 else f"{fmt_window(r['window_s'])}s"
    return f"{r['states']} · {r['atlas']} · {w}"

def save(fig, name):
    if SAVE_DIR:
        Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
        for ext in ("png", "svg"):
            fig.savefig(Path(SAVE_DIR) / f"{name}.{ext}", dpi=200, bbox_inches="tight")

def design(cov, intercept=True):
    """Covariates -> numeric matrix: categoricals one-hot, gaps mean-filled."""
    X = pd.get_dummies(cov, columns=[c for c in CATEGORICAL if c in cov],
                       drop_first=True).astype(float)
    X = X.fillna(X.mean()).to_numpy()
    return np.column_stack([np.ones(len(X)), X]) if intercept else X
TREE_COLOUR = {"bstm": "#2a78d6", "fcm": "#eb6834"}
TASK_COLOUR = dict(zip(TASKS, ["#1baf7a", "#6250d6"]))    # figures 1.2-1.6
TASK_MARKER = dict(zip(TASKS, ["o", "D"]))
DASHES = ["--", "-.", ":", (0, (6, 2, 1, 2))]             # one per model's baseline
# 8 validated hues, then two neutrals for slots 9-10. The marker shape is the
# second encoding, so no two sets rely on colour alone.
SET_COLOURS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4",
               "#008300", "#6250d6", "#e34948", "#52514e", "#8c6d4f"]
SET_MARKERS = ["o", "s", "^", "D", "v"]

In [ ]:
def read_summaries(tree):
    files = sorted((ROOT / f"{tree}_selection").glob("task=*/target=*/summary.csv"))
    if not files:
        raise SystemExit(f"no summary.csv under {ROOT / f'{tree}_selection'}")
    return pd.concat([pd.read_csv(p).assign(tree=tree, task=hive(p, "task"),
                                            target=hive(p, "target"))
                      for p in files], ignore_index=True)

def read_pheno(covariates):
    frames = [pd.read_csv(p, sep=s, dtype={PHENO_ID: str}) for p, s in PHENO_FILES]
    ph = frames[0]
    for f in frames[1:]:
        ph = ph.merge(f.drop(columns=[c for c in f if c in ph and c != PHENO_ID]),
                      on=PHENO_ID, how="outer")
    lower = {c.lower(): c for c in ph.columns}              # Age vs age
    ph = ph.rename(columns={lower[c.lower()]: c for c in covariates if c.lower() in lower})
    ph["sub"] = ph[PHENO_ID].astype(str).str.strip().str.upper()
    code = {lvl: k for k, lvl in enumerate(ORDINAL_LEVELS)}
    for lab, col in TARGETS.items():
        ph[lab] = ph[col].astype("string").str.strip().map(code).astype(float)
    for c in set(covariates) - set(CATEGORICAL) - {"n_transitions"}:
        ph[c] = pd.to_numeric(ph[c], errors="coerce")
    return ph[["sub", *TARGETS, *(c for c in covariates if c in ph)]]

def load_set(atlas, window_s, states, task):
    """One state set's transition table for one task, one row per subject."""
    base = (ROOT / "transitions" / f"atlas={atlas}" / f"window_s={fmt_window(window_s)}"
            / f"states={states}")
    files = [p for p in base.rglob("subjects.parquet")
             if hive(p, "cohort") in COHORTS
             and (hive(p, "task") or task).lower() == TASKS[task].lower()]
    if not files:
        raise SystemExit(f"no transition table for {base} task={task}")
    d = pd.concat(map(pd.read_parquet, files), ignore_index=True)
    d = d[d["task"].astype(str).str.lower() == TASKS[task].lower()]
    d["sub"] = d["sub"].astype(str).str.strip().str.upper()
    if d["sub"].duplicated().any():
        raise SystemExit(f"{states} {task}: a subject appears twice -- narrow COHORTS")
    return d

def cell_cols(d):
    return [c for c in d.columns if "->" in c]

def as_p_cond(d, K):
    """p_joint -> Markov p(j | i): back to counts, then renormalise each row."""
    cols = cell_cols(d)
    M = d[cols].to_numpy(float).reshape(len(d), K, K) * d["n_transitions"].to_numpy(float)[:, None, None]
    s = M.sum(2, keepdims=True)
    return pd.DataFrame(np.divide(M, s, out=np.zeros_like(M), where=s > 0).reshape(len(d), -1),
                        columns=cols, index=d.index)

def partial_spearman(A, y, cov):
    """Rank everything, residualise on the covariates, correlate. Returns r, p."""
    X = design(cov)
    R = np.column_stack([stats.rankdata(y), np.apply_along_axis(stats.rankdata, 0, A)])
    R = R - X @ np.linalg.lstsq(X, R, rcond=None)[0]
    R = (R - R.mean(0)) / np.where(R.std(0) > 0, R.std(0), np.nan)
    r = R[:, 1:].T @ R[:, 0] / len(R)
    df = len(R) - X.shape[1] - 1
    p = 2 * stats.t.sf(np.abs(r * np.sqrt(df / (1 - r ** 2))), df)
    return r, p

def top_sets(task, lab, n=TOP_N, arm=None):
    """The n best bstm state sets of one ranking (MODEL, `arm` or ARM), as summary.csv rows."""
    g = summ[(summ["tree"] == "bstm") & (summ["task"] == task) & (summ["phenotype"] == lab)
             & (summ["arm"] == (arm or ARM["bstm"])) & (summ["model"] == MODEL)]
    return (g.sort_values("mean", ascending=False)
             .drop_duplicates(["atlas", "window_s", "states"]).head(n).reset_index(drop=True))

# What each select-bstm arm hands the model, as in bstm_selection.build.
DYNAMICS = ["n_states_visited", "n_distinct_transitions", "switch_rate", "self_transition_rate",
            "switches_per_min", "mean_dwell_s", "entropy_rate_bits", "dispersion"]
ARM_BLOCKS = {"cells": ["cells"], "occupancy": ["occupancy"], "dynamics": ["dynamics"],
              "summary": ["occupancy", "dynamics"], "all": ["cells", "occupancy", "dynamics"]}

def arm_columns(t, arm):
    """{block: its columns in table t} for one arm, e.g. "all [cond]" -> cells, occupancy, dynamics."""
    cols = {"cells": cell_cols(t), "occupancy": [c for c in t.columns if c.startswith("occ_")],
            "dynamics": [c for c in DYNAMICS if c in t.columns]}
    return {b: cols[b] for b in ARM_BLOCKS[arm.split(" [")[0]]}

def merged(s, task, lab, arm=None):
    """One state set's table joined to phenotype, cells as the arm asks for ([cond] -> p(j | i))."""
    t = load_set(s["atlas"], s["window_s"], s["states"], task)
    if "[cond]" in (arm or ARM["bstm"]):
        t[cell_cols(t)] = as_p_cond(t, int(s["K"]))
    d = t.merge(pheno, on="sub", suffixes=("", "_pheno")).dropna(
        subset=[lab, *(c for c in COVARIATES if c not in CATEGORICAL)])
    return d, cell_cols(t)

label_of = {col: lab for lab, col in TARGETS.items()}
summ = pd.concat([read_summaries("bstm"), read_summaries("fcm")], ignore_index=True)
summ = summ[summ["target"].isin(label_of) & summ["task"].isin(TASKS)].assign(phenotype=lambda d: d["target"].map(label_of))
summ["label"] = summ.apply(set_label, axis=1)
summ["window"] = summ["window_s"].map(lambda w: None if pd.isna(w) else "1 TR" if w < 0 else f"{w:g}s")
summ[["method", "embedding", "K_"]] = summ["states"].str.rsplit("_", n=2, expand=True).reindex(columns=range(3))

pheno = read_pheno(COVARIATES + BASELINE_COVARIATES)
print(f"phenotype: {len(pheno):,} subjects;",
      ", ".join(f"{lab} {pheno[lab].value_counts().sort_index().astype(int).to_dict()}"
                for lab in TARGETS))

# Check this before reading a gap as a result: the runs are separate, so `n` can differ.
summ.pivot_table(index=["tree", "task"], columns="phenotype", values="n", aggfunc="max")

## 1. The four rankings

In [ ]:
import time
from joblib import Parallel, delayed

models = FIG1_MODELS or sorted(set(summ["model"]) - {"logistic"})
rows = summ[(summ["arm"] != BASELINE_ARM) & summ["model"].isin(models)
            & (summ["arm"].isin(FIG1_ARMS) if FIG1_ARMS else True)]
# one name per row that is the SAME row at rest and in the movie
rows = rows.assign(entity=rows["tree"] + " · " + rows["label"] + " · " + rows["arm"] + " · " + rows["model"])

def _fit(repo, X, y, model, seed):
    """One baseline score on ONE core. hgb otherwise spreads every tiny fit over all of a
    node's cores, which on a cluster is far slower than one core per fit side by side."""
    import sys
    from threadpoolctl import threadpool_limits
    sys.path.insert(0, repo)
    from fmri_decomposition.bstm_selection import score_once
    with threadpool_limits(1):
        return score_once(X, y, model, seed)

def baselines():
    """{phenotype: {model: mean, std, n}} -- every (phenotype, model, seed) fit in parallel."""
    data = {}
    for lab in TARGETS:                               # everyone the bstm runs scored
        subs = set().union(*(load_set(s["atlas"], s["window_s"], s["states"], task)["sub"]
                             for task in TASKS for _, s in top_sets(task, lab, 1).iterrows()))
        d = pheno[pheno["sub"].isin(subs)].dropna(
            subset=[lab, *(c for c in BASELINE_COVARIATES if c not in CATEGORICAL)])
        data[lab] = design(d[BASELINE_COVARIATES], intercept=False), d[lab].to_numpy(float)
    jobs = [(lab, m, s) for lab in TARGETS for m in models for s in SEEDS]
    scores = Parallel(n_jobs=min(N_JOBS, len(jobs)))(
        delayed(_fit)(str(REPO), *data[lab], m, s) for lab, m, s in jobs)
    out = {lab: {m: [] for m in models} for lab in TARGETS}
    for (lab, m, _), sc in zip(jobs, scores):
        out[lab][m].append(sc)
    return {lab: {m: {"mean": np.nanmean(v), "std": np.nanstd(v), "n": len(data[lab][1])}
                  for m, v in out[lab].items()} for lab in TARGETS}

# Its own cell, so re-drawing a figure does not refit.
t0 = time.time()
base = baselines()
print(f"{len(TARGETS) * len(models) * len(SEEDS)} baseline fits on {N_JOBS} core(s) in {time.time() - t0:.0f}s")
pd.DataFrame({(lab, m): b for lab in base for m, b in base[lab].items()}).T.round(3)

In [ ]:
RUNS = [(task, tree) for task in TASKS for tree in ("bstm", "fcm")]
ORIGIN_MARKER = dict(zip(TASKS, ["o", "D", "s", "^"]))   # which task a winner came from
BOTH_COLOURS = SET_COLOURS[2:]                            # blue and orange belong to the winners

# Each row is judged against ITS OWN model's baseline: every row already carries the
# covariates, so a shared bar would mostly select whichever model fits Age + Sex best.
_key = list(zip(rows["phenotype"], rows["model"]))
rows = rows.assign(base_mean=[base[l][m]["mean"] for l, m in _key],
                   base_sd=[base[l][m]["std"] for l, m in _key])

def draw_band(ax, b, x=None):
    """One model's Age + Sex baseline: dashed mean, ± sd band (full width, or over x = (lo, hi))."""
    lo, hi = x or ax.get_xlim()
    ax.fill_between([lo, hi], b["mean"] - b["std"], b["mean"] + b["std"], color=MUTED, alpha=.18, lw=0)
    ax.plot([lo, hi], [b["mean"]] * 2, color=INK2, ls="--", lw=.9)
    ax.axhline(0, color=MUTED, lw=.6)

def above_both(d):
    """Rows above their model's baseline + sd in EVERY task, best worst-task margin first."""
    margin = d["mean"] - MARGIN_SD * d["std"] - (d["base_mean"] + d["base_sd"])
    worst = d.assign(margin=margin)[margin > 0].groupby("entity").agg(
        tasks=("task", "nunique"), worst=("margin", "min"))
    return worst[worst["tasks"] == len(TASKS)].sort_values("worst", ascending=False).index.tolist()

def winners(d, tree):
    """{task: that tree's best row in that task}."""
    t = d[d["tree"] == tree]
    return {task: g.loc[g["mean"].idxmax()] for task, g in t.groupby("task", sort=False)}

def plot_runs(ax, d, b):
    rng = np.random.default_rng(0)
    keep = above_both(d)
    shown = keep[:HIGHLIGHT_MAX]
    style = {e: (BOTH_COLOURS[k % len(BOTH_COLOURS)], SET_MARKERS[k // len(BOTH_COLOURS) % len(SET_MARKERS)])
             for k, e in enumerate(shown)}
    offset = dict(zip(shown, np.linspace(-.3, .3, len(shown)) if len(shown) > 1 else [0]))
    win = {tree: winners(d, tree) for tree in ("bstm", "fcm")}
    ax.set_xlim(-.6, len(RUNS) - .4)
    draw_band(ax, b)
    for i, (task, tree) in enumerate(RUNS):
        g, c = d[(d["task"] == task) & (d["tree"] == tree)], TREE_COLOUR[tree]
        ax.scatter(i + rng.uniform(-.14, .14, len(g)), g["mean"], s=10, color=c, alpha=.2, lw=0)
        for r in g[g["entity"].isin(style)].itertuples():       # same x offset in both tasks
            cc, m = style[r.entity]
            ax.errorbar(i + offset[r.entity], r.mean, yerr=r.std, fmt=m, ms=6, lw=1.2,
                        color=cc, mec=SURFACE, mew=.8, zorder=3)
        for origin, w in win[tree].items():                     # filled where it won, hollow carried over
            r = g[g["entity"] == w["entity"]]
            if r.empty:
                continue
            own = origin == task
            ax.errorbar(i - .38 if own else i + .38, r["mean"].iloc[0], yerr=r["std"].iloc[0],
                        fmt=ORIGIN_MARKER[origin], ms=8, lw=1.5, color=c, zorder=4,
                        mfc=c if own else SURFACE, mec=SURFACE if own else c, mew=1.5)
    ax.set_xticks(range(len(RUNS)), [f"{tree}\n{task}" for task, tree in RUNS])
    for k in range(1, len(TASKS)):
        ax.axvline(2 * k - .5, color=MUTED, lw=.6)

    handles = [Line2D([], [], ls="none", marker=ORIGIN_MARKER[o], ms=7, color=TREE_COLOUR[tree],
                      label=f"{tree} best at {o}:  {w['label']} · {w['arm']}")
               for tree in ("bstm", "fcm") for o, w in win[tree].items()]
    handles += [Line2D([], [], ls="none", marker=style[e][1], ms=6, color=style[e][0], label=e)
                for e in shown]
    if len(keep) > len(shown):
        handles.append(Line2D([], [], ls="none", label=f"… and {len(keep) - len(shown)} more (table below)"))
    handles.append(Patch(color=MUTED, alpha=.35, label=
        f"Age + Sex baseline:  {b['mean']:.3f} ± {b['std']:.3f}  (n={b['n']})"))
    return handles, (f"large = best at each task: filled where it won, hollow carried over\n"
                     f"coloured = {len(keep)} row(s) above the baseline at {' AND '.join(TASKS)} (best {len(shown)} shown)")

# One block per model: its panels on top, their legends underneath. All plots share one y scale.
n_lines = HIGHLIGHT_MAX + 2 * len(TASKS) + 3
fig, axes = plt.subplots(2 * len(models), len(TARGETS), squeeze=False,
                         figsize=(6.5 * len(TARGETS), len(models) * (4 + .16 * n_lines)),
                         gridspec_kw={"height_ratios": [4, .16 * n_lines] * len(models)})
for k, model in enumerate(models):
    for j, lab in enumerate(TARGETS):
        ax, leg = axes[2 * k, j], axes[2 * k + 1, j]
        if (k, j) != (0, 0):
            ax.sharey(axes[0, 0])
        handles, title = plot_runs(ax, rows[(rows["phenotype"] == lab) & (rows["model"] == model)],
                                   base[lab][model])
        ax.set_title(f"{model} · {lab}")
        leg.axis("off")
        leg.legend(handles=handles, loc="upper center", fontsize=7, title=title, title_fontsize=7)
    axes[2 * k, 0].set_ylabel("CV score (Spearman)")
fig.suptitle("every row · dashed = that model's Age + Sex baseline, band ± sd · coloured = above it at "
             f"{' and '.join(TASKS)}, ± sd over seeds", fontsize=8, color=INK2, y=.995)
fig.tight_layout()
save(fig, "fig1_rankings")

In [ ]:
# Every row above its model's baseline at both tasks, by model (best worst-task margin first).
pd.concat({(lab, m): (lambda d: d[d["entity"].isin(above_both(d))]
                      .pivot_table(index=["tree", "arm", "label"], columns="task", values="mean")
                      .reindex(columns=list(TASKS))
                      .pipe(lambda t: t.loc[t.min(axis=1).sort_values(ascending=False).index])
                      .assign(baseline=base[lab][m]["mean"], baseline_sd=base[lab][m]["std"]))(
               rows[(rows["phenotype"] == lab) & (rows["model"] == m)])
           for lab in TARGETS for m in models}, names=["phenotype", "model"]).round(3)

## 1.2–1.6 What drives the score

One figure per axis of the grid (`FACTORS`), on the same rows as figure 1 (`FACTOR_TREES`,
bstm by default). There is one row of panels per model, each with that model's baseline, because a
ridge row and an hgb row stand on different baselines. Every row of the grid is a faint dot, put
in the group for its window, atlas, state method and so on. Rest and movie sit side by side in each
group. The marker is the median, and the bar spans the middle half (25th–75th percentile).
1.4 (model) puts the models in one panel instead, each over its own baseline band.

A group is pooled over everything else in the grid. If one window size only ever ran with a
strong atlas, it inherits that atlas's scores, so read a gap between groups as a lead to check
in figure 1, not as an effect.

In [ ]:
NUMERIC_ORDER = {"window": "window_s", "K_": "K"}       # ordered by size; the rest by median score,
                                                        # once per figure so all panels match

def levels(d, col):
    if col in NUMERIC_ORDER:
        return d.groupby(col)[NUMERIC_ORDER[col]].first().sort_values().index.tolist()
    return d.groupby(col)["mean"].median().sort_values(ascending=False).index.tolist()

def compare(ax, d, col, bases, lev):
    """One panel: rows grouped by `col`, rest and movie side by side, over the baseline(s)."""
    d = d.dropna(subset=[col])
    rng = np.random.default_rng(0)
    ax.set_xlim(-.6, len(lev) - .4)
    if col == "model":                                   # each model over its own band
        for i, v in enumerate(lev):
            draw_band(ax, bases[v], (i - .45, i + .45))
    else:
        draw_band(ax, next(iter(bases.values())))
    for j, task in enumerate(TASKS):
        dx, c = (j - (len(TASKS) - 1) / 2) * .3, TASK_COLOUR[task]
        for i, v in enumerate(lev):
            s = d.loc[(d[col] == v) & (d["task"] == task), "mean"]
            if s.empty:
                continue
            ax.scatter(i + dx + rng.uniform(-.06, .06, len(s)), s, s=8, color=c, alpha=.25, lw=0)
            q1, q2, q3 = np.percentile(s, [25, 50, 75])
            ax.plot([i + dx] * 2, [q1, q3], color=c, lw=3, solid_capstyle="round", zorder=3)
            ax.plot(i + dx, q2, TASK_MARKER[task], ms=7, color=c, mec=SURFACE, mew=1, zorder=4)
    long = max(map(len, map(str, lev))) > 8
    ax.set_xticks(range(len(lev)), lev, rotation=30 if long else 0, ha="right" if long else "center")

pool = rows[rows["tree"].isin(FACTOR_TREES)]
for k, (col, name) in enumerate(FACTORS.items(), start=2):
    blocks = [None] if col == "model" else models        # None = every model in one panel
    fig, axes = plt.subplots(len(blocks), len(TARGETS), squeeze=False, sharey=True,
                             figsize=(6.5 * len(TARGETS), 3.6 * len(blocks) + .4))
    lev = levels(pool.dropna(subset=[col]), col)
    for r, model in enumerate(blocks):
        for ax, lab in zip(axes[r], TARGETS):
            d = pool[(pool["phenotype"] == lab) & ((pool["model"] == model) if model else True)]
            compare(ax, d, col, {m: base[lab][m] for m in ([model] if model else models)}, lev)
            ax.set_title(f"{model} · {lab}" if model else lab)
        axes[r, 0].set_ylabel("CV score (Spearman)")
    handles = [Line2D([], [], ls="none", marker=TASK_MARKER[t], ms=7, color=TASK_COLOUR[t],
                      label=f"{t}: median, bar = middle half") for t in TASKS]
    fig.legend(handles=handles + [Patch(color=MUTED, alpha=.35, label="that model's Age + Sex baseline ± sd")],
               loc="upper right", ncol=3, fontsize=7, bbox_to_anchor=(1, 1.01))
    fig.suptitle(f"1.{k}  {name}  ·  {', '.join(FACTOR_TREES)}", x=.01, ha="left", fontsize=10,
                 color=INK, y=1.0)
    fig.tight_layout()
    save(fig, f"fig1_{k}_{col}")
    plt.show()

## 2. Volcano with an effect size on x

**Why a correlation volcano (`X_AXIS = "r"`) is a narrow V.** A correlation already divides out each
cell's noise. With n fixed, p depends on |r| alone, so every dot lands on one curve. A gene volcano puts
a *raw* effect on x (log2 fold change) and takes p from a test that uses each feature's own variance,
so two features with the same fold change can have very different p.

**Why the raw fold change is too wide.** A ratio of two near-zero means is mostly noise. A rare cell
that moves from 0.0004 to 0.0010 is a 1.3 log2 fold change and means nothing. `SHRINK` adds a pseudocount
to both means: `log2((elevated + c) / (Normal + c))`, where `c = SHRINK × the average cell`.
Common cells barely move, and rare cells are pulled toward 0. This is the same idea as the
shrunken fold changes in DESeq2. `SHRINK = 0` gives the raw ratio. Raising it narrows the cloud
toward a slope-like scale. p is unaffected: it comes from the t-test of the group term in a
regression on the covariates.

There is one panel for each (task, phenotype) pair. Each panel draws its `TOP_N` sets from its own ranking,
so the colours in different panels belong to different sets: read the legend under each panel.
Dots are `(state set, i->j)` cells, and filled dots with a dark ring survive BH-FDR over every cell in the panel.
The **dashed** line is that FDR cut (the p of the weakest surviving cell). The **dotted** line is p = 0.05 uncorrected.
The cells are compositional, because each subject's row sums to 1, so hits tend to come in sprays.
Read the strongest cell and treat its neighbours as the same effect.

In [ ]:
CASE = ORDINAL_LEVELS.index(CASE_FROM)
XLABEL = {"log2fc": f"log2 fold change  ({CASE_FROM}+ vs Normal, adjusted, shrink={SHRINK:g})",
          "slope": "Δ per HADS level  (adjusted, in each variable's own units)",
          "r": f"partial Spearman r  (| {', '.join(COVARIATES)})"}[X_AXIS]
def passes(v):
    """Highlighted = survives FDR AND clears the effect-size cut (|x| >= FC_LINE), as in a gene volcano."""
    return (v["p_fdr"] < ALPHA) & (v["x"].abs() >= (FC_LINE or 0))

PASS_TEXT = f"FDR<{ALPHA:g}" + (f" & |x|≥{FC_LINE:g}" if FC_LINE else "")
PROBABILITY = {"cells", "occupancy"}       # blocks that take MIN_PRESENT and the pseudocount

def effects(A, y, cov, c=0.0, group=None):
    """Effect of the phenotype on every column of A, adjusted for cov. Returns (x, p).
    `c` is the log2fc pseudocount; 0 for variables that are never near zero. `group` (0/1)
    replaces y with a ready-made comparison, e.g. a symptom group vs the Normal group."""
    if X_AXIS == "r":
        return partial_spearman(A, y if group is None else group, cov)
    g = group if group is not None else (y >= CASE).astype(float) if X_AXIS == "log2fc" else y
    X = np.column_stack([g, design(cov)])
    B = np.linalg.lstsq(X, A, rcond=None)[0]
    df = len(A) - X.shape[1]
    se = np.sqrt(((A - X @ B) ** 2).sum(0) / df * np.linalg.pinv(X.T @ X)[0, 0])
    p = 2 * stats.t.sf(np.abs(B[0] / se), df)
    if X_AXIS == "slope":
        return B[0], p
    mu0 = A[g == 0].mean(0)                           # Normal
    mu1 = np.clip(mu0 + B[0], 0, None)                # elevated, adjusted; a mean cannot go below 0
    return np.log2((mu1 + c + 1e-12) / (mu0 + c + 1e-12)), p

def test_blocks(d, lab, arm):
    """Every variable of `arm` in one joined table vs one phenotype: one frame per block."""
    out = []
    for block, cols in arm_columns(d, arm).items():
        A = d[cols].astype(float)
        A = A.loc[:, A.std() > 0]                                     # flat: untestable
        if block in PROBABILITY:
            A = A.loc[:, (A > 0).mean() >= MIN_PRESENT]               # rare: a ratio of noise
        if A.shape[1] == 0:
            continue
        # pseudocount: the average cell for probabilities (pulls rare cells in); each variable's
        # own mean for dynamics (bounds the ratio when one sits near zero, e.g. switch_rate at 1 TR)
        c = SHRINK * (A.to_numpy().mean() if block in PROBABILITY else A.mean().to_numpy())
        x, p = effects(A.to_numpy(), d[lab].to_numpy(float), d[COVARIATES], c)
        out.append(pd.DataFrame({"block": block, "variable": A.columns, "x": x, "p": p, "n": len(d)}))
    return out

def scan(task, lab, arm, rank_arm=None):
    """Every variable of `arm` in the top sets of `arm`'s ranking vs one phenotype. One row per variable."""
    sets = top_sets(task, lab, arm=rank_arm or arm)
    if sets.empty:
        raise SystemExit(f"no {MODEL} rows for arm {arm!r} in bstm {task} {lab}; "
                         f"arms there: {sorted(summ.loc[summ['tree'] == 'bstm', 'arm'].unique())}")
    v = pd.concat([r.assign(rank=rank + 1, set=s["label"], score=s["mean"])
                   for rank, s in sets.iterrows()
                   for r in test_blocks(merged(s, task, lab, arm)[0], lab, arm)], ignore_index=True)
    v["p_fdr"] = stats.false_discovery_control(v["p"], method="bh")
    return v.assign(task=task, phenotype=lab)

volc = pd.concat([scan(task, lab, ARM["bstm"]) for task in TASKS for lab in TARGETS], ignore_index=True)
volc.nsmallest(10, "p")[["task", "phenotype", "rank", "set", "variable", "x", "p", "p_fdr", "n"]]

In [ ]:
BLOCK_MARKER = {"cells": "o", "occupancy": "s", "dynamics": "^"}

def label_points(ax, v, name):
    """Name every FDR survivor (or the LABEL_TOP smallest p if none) in a column right of the
    panel, top to bottom in the points' order, with a thin line to each: names never overlap."""
    t = v[passes(v)].nsmallest(LABEL_MAX, "p")
    t = (t if len(t) else v.nsmallest(LABEL_TOP, "p")).copy()
    if t.empty:
        return
    t["y"] = -np.log10(t["p"])
    lo, hi = ax.get_ylim()
    t["f"] = ((t["y"] - lo) / (hi - lo)).clip(0, 1)              # height as a fraction of the panel
    t = t.sort_values("f", ascending=False)
    gap = min(.035, .98 / len(t))
    pos, cur = [], 1.0
    for f in t["f"]:                                            # as close to the point as the one above allows
        cur = min(f, cur - gap) if pos else min(f, 1.0)
        pos.append(cur)
    floor = 0.0                                                 # second pass, bottom-up: nothing below 0
    for i in range(len(pos) - 1, -1, -1):
        pos[i] = max(pos[i], floor)
        floor = pos[i] + gap
    for r, yl in zip(t.itertuples(), pos):
        ax.annotate(name(r), (r.x, r.y), xytext=(1.02, yl), textcoords="axes fraction",
                    fontsize=6.5, color=INK2, va="center", annotation_clip=False,
                    arrowprops=dict(arrowstyle="-", lw=.4, color=MUTED, shrinkA=0, shrinkB=2,
                                    relpos=(0, .5)))       # leave from the start of the name

def plot_volcano(ax, v):
    """Colour = state set. Marker = set too, or the variable type when there is more than cells."""
    sig, fdr = passes(v), v["p_fdr"] < ALPHA
    by_block = set(v["block"]) != {"cells"}
    handles = []
    for k, (rank, g) in enumerate(v.groupby("rank")):
        c = SET_COLOURS[k % len(SET_COLOURS)]
        for block, h in g.groupby("block"):
            m = BLOCK_MARKER[block] if by_block else SET_MARKERS[k % len(SET_MARKERS)]
            y, hit = -np.log10(h["p"]), sig[h.index]
            ax.scatter(h["x"], y, s=14, marker=m, color=c, alpha=.45, lw=0)
            ax.scatter(h["x"][hit], y[hit], s=40, marker=m, color=c, edgecolor=INK, lw=.8)
        handles.append(Line2D([], [], ls="none", ms=5, color=c, label=f"#{rank}  {g['set'].iloc[0]}",
                              marker="o" if by_block else SET_MARKERS[k % len(SET_MARKERS)]))
    if by_block:
        handles += [Line2D([], [], ls="none", ms=5, marker=BLOCK_MARKER[b], color=INK2, label=b)
                    for b in BLOCK_MARKER if b in set(v["block"])]
    ax.axhline(-np.log10(.05), color=MUTED, ls=":", lw=.8)
    if fdr.any():                                         # the FDR cut itself, whatever the effect size
        ax.axhline(-np.log10(v.loc[fdr, "p"].max()), color=INK2, ls="--", lw=.8)
    ax.axvline(0, color=MUTED, lw=.6)
    for side in (-1, 1) if FC_LINE else ():               # the effect-size cut, as in a gene volcano
        ax.axvline(side * FC_LINE, color=INK2, ls="--", lw=.8)
    lim = XLIM or 1.08 * max(v["x"].abs().max(), 1.1 * (FC_LINE or 0))
    ax.set_xlim(-lim, lim)
    out = int((v["x"].abs() > lim).sum())
    if out:                                               # beyond a fixed XLIM: say so, do not hide it
        ax.text(.99, .01, f"{out} beyond ±{lim:g}", transform=ax.transAxes, ha="right", fontsize=7, color=INK2)
    label_points(ax, v, lambda t: f"{t.variable} (#{t.rank})")
    ax.set_title(f"{v['task'].iloc[0]} · {v['phenotype'].iloc[0]}   "
                 f"({sig.sum()} / {len(v):,} variables {PASS_TEXT}, n≈{int(v['n'].median())})",
                 fontsize=10)
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(.5, -.2), ncol=2, fontsize=7,
              handletextpad=.3, columnspacing=1, markerscale=1.3)

def draw_volcanos(volc, name, title=None):
    """The 2 x 2: rest and movie by anxiety and depression."""
    fig, axes = plt.subplots(len(TASKS), len(TARGETS), squeeze=False,
                             figsize=(8.5 * len(TARGETS), 6.3 * len(TASKS)))
    for (i, task), (j, lab) in ((a, b) for a in enumerate(TASKS) for b in enumerate(TARGETS)):
        plot_volcano(axes[i, j], volc[(volc["task"] == task) & (volc["phenotype"] == lab)])
    for ax in axes[:, 0]:
        ax.set_ylabel(r"$-\log_{10}$ p")
    for ax in axes[-1]:
        ax.set_xlabel(XLABEL)
    if title:
        fig.suptitle(title, x=.01, ha="left", fontsize=10, color=INK)
    fig.tight_layout(h_pad=3, rect=(0, 0, 1, .975 if title else 1))
    save(fig, name)
    plt.show()

draw_volcanos(volc, f"fig2_volcano_{X_AXIS}")

## 2.1 Volcano over every variable of an arm

The same 2 × 2 as figure 2, but the variables come from `ARM_21`, not only the transition cells.
Each `transitions/` table also stores the occupancy (`occ_k`, the share of time in state k) and
eight dynamics scalars (`switch_rate`, `self_transition_rate`, `switches_per_min`, `mean_dwell_s`,
`entropy_rate_bits`, `dispersion`, `n_states_visited`, `n_distinct_transitions`). `select-bstm`'s
arms are built from exactly these:

| `ARM_21` | variables |
|---|---|
| `cells [joint]` / `cells [cond]` | the K×K transitions, as p_joint / as p(j \| i) |
| `occupancy` | `occ_0 … occ_{K-1}` |
| `dynamics` | the eight scalars |
| `summary` | occupancy + dynamics |
| `all [joint]` / `all [cond]` | all three |

The `TOP_N` sets come from `ARM_21`'s own ranking (same `MODEL`), so the panels show the sets
where that arm predicted best. Colour shows the set and the marker shows the variable type.
FDR runs over every variable in the panel.

**2.1 is not a superset of figure 2 by default.** It ranks with another arm, so it shows other
sets, and `[cond]` cells are p(j | i), not figure 2's p_joint. Its FDR family is also larger.
`RANK_21 = ARM["bstm"]` with `ARM_21 = "all [joint]"` keeps figure 2's sets and cells and adds
occupancy and dynamics, so figure 2's dots reappear at the same x and p. Only the FDR line moves.
Figure 3 with `ARM_3 = ARM_21` contains every dot of 2.1 the same way.

On the x axis, the log2fc pseudocount for probabilities (cells and occupancy) is `SHRINK` × the
average cell, and `MIN_PRESENT` applies to them only. For dynamics it is `SHRINK` × that variable's
own mean, which keeps a variable near zero from producing an unbounded ratio. `X_AXIS = "slope"`
puts each variable in its own units (dwell in seconds, entropy in bits), which do not compare
across types: prefer `log2fc` or `r` here. `switch_rate` and `self_transition_rate` sum to 1, so
they mirror each other.

In [ ]:
volc21 = pd.concat([scan(task, lab, ARM_21, RANK_21) for task in TASKS for lab in TARGETS], ignore_index=True)
draw_volcanos(volc21, "fig2_1_volcano_" + ARM_21.replace(" ", "").replace("[", "_").strip("]"),
              title=f"2.1  every variable of `{ARM_21}` · top {TOP_N} sets of its {MODEL} ranking")
volc21.nsmallest(10, "p")[["task", "phenotype", "rank", "set", "block", "variable", "x", "p", "p_fdr"]]

## 3. The multiverse: every variable of every state set

No ranking and no top 10. Every `subjects.parquet` under `transitions/` (each atlas × window ×
state definition), every variable of `ARM_3`: one dot is one variable in one set, e.g.
`entropy_rate_bits · harvardoxford · 30s · HMM1_pca3_8`. The panels are the usual rest/movie ×
anxiety/depression, and the test is figure 2's (`X_AXIS`, adjusted for `COVARIATES`).

The point is consistency. A dynamics variable whose dots stay on one side of zero, whatever the
atlas, window or method, does not depend on those choices. A hit that shows up in only one corner
of the grid does. Only the dynamics mean the same thing in every set. State 3 in one set has
nothing to do with state 3 in another, so for cells and occupancy read the cloud as a whole, not
dot by dot.

FDR (BH) runs over every dot in a panel. That is tens of thousands of strongly correlated tests,
so it is strict; BH stays valid under that kind of positive dependence. Dots that survive are
drawn solid and the rest faint. `COLOR_BY_3` gives one figure per colouring.

In [ ]:
def all_sets():
    """Every (atlas, window_s, states) with a table under transitions/, for COHORTS."""
    found = {(hive(p, "atlas"), float(hive(p, "window_s")), hive(p, "states"))
             for p in (ROOT / "transitions").rglob("subjects.parquet") if hive(p, "cohort") in COHORTS}
    return pd.DataFrame(sorted(found), columns=["atlas", "window_s", "states"]).assign(
        K=lambda d: d["states"].str.rsplit("_", n=1).str[1].astype(int))

def scan_set(s, task):
    """One table, read once, every variable of ARM_3 vs every phenotype."""
    try:
        t = load_set(s["atlas"], s["window_s"], s["states"], task)
    except SystemExit:                                  # this set has no table for this task
        return []
    if "[cond]" in ARM_3:
        t[cell_cols(t)] = as_p_cond(t, int(s["K"]))
    d0 = t.merge(pheno, on="sub", suffixes=("", "_pheno"))
    out = []
    for lab in TARGETS:
        d = d0.dropna(subset=[lab, *(c for c in COVARIATES if c not in CATEGORICAL)])
        out += [r.assign(atlas=s["atlas"], window_s=s["window_s"], states=s["states"], K=s["K"],
                         task=task, phenotype=lab)
                for r in test_blocks(d, lab, ARM_3)]
    return out

sets3 = all_sets()
t0 = time.time()
parts = Parallel(n_jobs=N_JOBS, prefer="threads")(               # reading parquet + numpy: threads are enough
    delayed(scan_set)(s, task) for _, s in sets3.iterrows() for task in TASKS)
multi = pd.concat([r for part in parts for r in part], ignore_index=True)
multi["p_fdr"] = multi.groupby(["task", "phenotype"])["p"].transform(
    lambda p: stats.false_discovery_control(p, method="bh"))
multi["hit"] = passes(multi)
multi["window"] = multi["window_s"].map(lambda w: "1 TR" if w < 0 else f"{w:g}s")
multi["method"] = multi["states"].str.rsplit("_", n=2).str[0]
multi["set"] = multi["atlas"] + " · " + multi["window"] + " · " + multi["states"]
print(f"{len(sets3)} state sets, {len(multi):,} dots in {time.time() - t0:.0f}s")
multi.groupby(["task", "phenotype"]).agg(
    dots=("p", "size"), sets=("set", "nunique"),
    hits=("hit", "sum"))

In [ ]:
RAMP = ["#86b6ef", "#6da7ec", "#5598e7", "#3987e5", "#2a78d6",   # sequential blue, light -> dark,
        "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]   # for the ordered axes
ORDERED = {"window": "window_s", "K": "K"}

def palette(v, by):
    """{level: colour}: sequential for window and K (they have an order), categorical otherwise."""
    if by in ORDERED:
        lev = v.groupby(by)[ORDERED[by]].first().sort_values().index.tolist()
        idx = np.linspace(0, len(RAMP) - 1, len(lev)).round().astype(int)
        return dict(zip(lev, [RAMP[i] for i in idx]))
    lev = v[by].value_counts().index.tolist()
    return {l: SET_COLOURS[k % len(SET_COLOURS)] for k, l in enumerate(lev)}

def plot_multiverse(ax, v, by, colours):
    v = v.sample(frac=1, random_state=0)                 # no group always drawn on top
    y, sig, fdr, c = -np.log10(v["p"]), passes(v), v["p_fdr"] < ALPHA, v[by].map(colours)
    ax.scatter(v["x"][~sig], y[~sig], s=4, c=c[~sig], alpha=.25, lw=0, rasterized=True)
    ax.scatter(v["x"][sig], y[sig], s=12, c=c[sig], alpha=.95, lw=0, rasterized=True)
    ax.axhline(-np.log10(.05), color=MUTED, ls=":", lw=.8)
    if fdr.any():                                         # the FDR cut itself, whatever the effect size
        ax.axhline(-np.log10(v.loc[fdr, "p"].max()), color=INK2, ls="--", lw=.8)
    ax.axvline(0, color=MUTED, lw=.6)
    for side in (-1, 1) if FC_LINE else ():               # the effect-size cut, as in a gene volcano
        ax.axvline(side * FC_LINE, color=INK2, ls="--", lw=.8)
    lim = XLIM or 1.08 * max(v["x"].abs().max(), 1.1 * (FC_LINE or 0))
    ax.set_xlim(-lim, lim)
    out = int((v["x"].abs() > lim).sum())
    if out:                                               # beyond a fixed XLIM: say so, do not hide it
        ax.text(.99, .01, f"{out} beyond ±{lim:g}", transform=ax.transAxes, ha="right", fontsize=7, color=INK2)
    label_points(ax, v, lambda t: f"{t.variable} · {t.atlas} · {t.window} · {t.states}")
    ax.set_title(f"{v['task'].iloc[0]} · {v['phenotype'].iloc[0]}   "
                 f"({sig.sum():,} / {len(v):,} {PASS_TEXT})", fontsize=10)

for by in COLOR_BY_3:
    colours = palette(multi, by)
    fig, axes = plt.subplots(len(TASKS), len(TARGETS), squeeze=False,
                             figsize=(9.5 * len(TARGETS), 5.2 * len(TASKS)))
    for (i, task), (j, lab) in ((a, b) for a in enumerate(TASKS) for b in enumerate(TARGETS)):
        plot_multiverse(axes[i, j], multi[(multi["task"] == task) & (multi["phenotype"] == lab)], by, colours)
    for ax in axes[:, 0]:
        ax.set_ylabel(r"$-\log_{10}$ p")
    for ax in axes[-1]:
        ax.set_xlabel(XLABEL)
    fig.legend(handles=[Line2D([], [], ls="none", marker="o", ms=6, color=c, label=str(l))
                        for l, c in colours.items()] +
                       [Line2D([], [], ls="none", label=f"solid = {PASS_TEXT} (FDR over the panel)")],
               loc="upper right", ncol=min(len(colours) + 1, 8), fontsize=7, bbox_to_anchor=(1, 1.0))
    fig.suptitle(f"3  multiverse · every variable of `{ARM_3}` in every state set · colour = {by}",
                 x=.01, ha="left", fontsize=10, color=INK)
    fig.tight_layout(h_pad=2.5, rect=(0, 0, 1, .96))
    save(fig, f"fig3_multiverse_{by}")
    plt.show()

In [ ]:
# The dynamics are the variables that mean the same thing in every set: how consistent is each one?
# share_up = share of sets where it is higher with more symptoms; hits = sets where it survives FDR.
dyn = multi[multi["block"] == "dynamics"]
(dyn.groupby(["phenotype", "variable", "task"])
    .agg(sets=("x", "size"), share_up=("x", lambda x: (x > 0).mean()), median_x=("x", "median"),
         hits=("hit", "sum"))
    .unstack("task").round(3))

## 4. How the brain dynamics differ: the volcano drawn on the state graph

One state set (`SET_4`, by default the one whose `MODEL` / `ARM` score is best averaged over
the four panels), one row per task, three diagrams per row. **Normal** means Normal on both HADS
scales and is the reference. **Anxiety** and **depression** mean `CASE_FROM` or above on that
scale; someone can be in both. Each diagram is that group's own dynamics, pooled over its subjects.

* **Circle area** is the time spent in the state, with the % beside it. **The loop** is p(stay),
  thicker when the state is held longer. **Arrow width** is p(j | i), the share of moves out of i
  that go to j. A row draws the same arrows (the Normal group's `EDGE_TOP` strongest plus every
  red one) on one width scale.
* **Red marks what differs from Normal by the volcano's own rule.** Each subject's p(j | i),
  p(stay) and occupancy is tested group vs Normal, adjusted for `COVARIATES`, with the effect on
  the `X_AXIS` scale. Red means the point passes FDR and |x| ≥ FC_LINE, with BH-FDR over everything in the diagram.
* **Where the numbers are.** The % beside each state is the time spent in it (the circle), and a
  red one carries its change vs Normal, e.g. `45% (-16 pp)`. A red loop's change in p(stay) sits at
  the loop's corner, e.g. `-20 pp`, and a red arrow's change in p(j | i) next to the middle of the arrow, e.g.
  `+21 pp`. Time in a state and p(stay) move together but are different numbers: leaving a state
  more often lowers p(stay) first, and the time in it only as a consequence.

In [ ]:
import matplotlib
from matplotlib.patches import Arc, Circle, FancyArrowPatch

HIGHLIGHT, GREY, NODE = "#e34948", "#a3a29d", "#ecebe7"
CASE = ORDINAL_LEVELS.index(CASE_FROM)

def pct(v):
    """A share as %, never rounded to a misleading 0: a rarely visited state still has transitions."""
    return "0%" if v == 0 else "<0.1%" if v < .001 else f"{100 * v:.1f}%" if v < .01 else f"{100 * v:.0f}%"

def away(direction):
    """ha / va that make a label grow AWAY from its anchor along `direction` (never back over it)."""
    x, y = direction / np.linalg.norm(direction)
    return ("left" if x > .35 else "right" if x < -.35 else "center",
            "bottom" if y > .35 else "top" if y < -.35 else "center")

def clear_of(xy, text, centres, radii, scale):
    """True if a label centred at xy (box ~ its text, in data units) touches no circle."""
    hw, hh = .5 * len(text) * .042 * scale + .02, .05 * scale            # 6.5 pt text on a ~4.6 in panel
    dx = np.maximum(np.abs(centres[:, 0] - xy[0]) - hw, 0)
    dy = np.maximum(np.abs(centres[:, 1] - xy[1]) - hh, 0)
    return bool((dx ** 2 + dy ** 2 > (radii + .015) ** 2).all())

def state_table(s, task):
    """One state set's per-subject table for one task, joined to both phenotypes."""
    t = load_set(s["atlas"], s["window_s"], s["states"], task)
    return t.merge(pheno, on="sub", suffixes=("", "_pheno")).dropna(
        subset=[*TARGETS, *(c for c in COVARIATES if c not in CATEGORICAL)]).reset_index(drop=True)

def groups(d):
    """Normal = Normal on BOTH scales (the reference); each phenotype = CASE_FROM or above on it."""
    return {"Normal": (d[list(TARGETS)] == 0).all(axis=1).to_numpy(),
            **{lab: (d[lab] >= CASE).to_numpy() for lab in TARGETS}}

def pooled(d, K, m):
    """A group's p(j | i), pooled over its subjects' transition counts, and its mean occupancy."""
    c = (d.loc[m, cell_cols(d)].to_numpy(float) * d.loc[m, "n_transitions"].to_numpy(float)[:, None]).sum(0).reshape(K, K)
    rows = c.sum(1, keepdims=True)
    return c / np.where(rows > 0, rows, 1), d.loc[m, [f"occ_{k}" for k in range(K)]].to_numpy(float).mean(0)

def diagram(ax, T, occ, edges, wmax, omax, ref=None, hit_T=None, hit_occ=None, note_T=None, note_occ=None):
    """A state diagram on a circle. Circle area = time in state, loop = p(stay), arrow width =
    p(j | i). `ref` = (T, occ) of the reference group; when given, the diagram is a comparison and
    everything not highlighted is drawn in grey.
    hit_T / hit_occ are drawn in HIGHLIGHT, with note_T / note_occ written beside them."""
    K = len(occ)
    hit_T = np.zeros((K, K), bool) if hit_T is None else hit_T
    hit_occ = np.zeros(K, bool) if hit_occ is None else hit_occ
    ang = np.pi / 2 - 2 * np.pi * np.arange(K) / K
    pos = np.column_stack([np.cos(ang), np.sin(ang)])
    radius = lambda o: .05 + .17 * np.sqrt(o / omax)                    # area ~ time in state
    base = INK2 if ref is None else GREY
    radii = np.array([radius(o) for o in occ])
    nodes = [ax.add_patch(Circle(pos[k], radius(occ[k]), zorder=4, lw=1.2 if hit_occ[k] else .7,
                                 fc=matplotlib.colors.to_rgba(HIGHLIGHT, .3) if hit_occ[k] else NODE,
                                 ec=HIGHLIGHT if hit_occ[k] else INK2)) for k in range(K)]
    for i, j in edges:                                                  # transitions
        hit = hit_T[i, j]
        ax.add_patch(FancyArrowPatch(pos[i], pos[j], connectionstyle="arc3,rad=.18", arrowstyle="-|>",
                                     patchA=nodes[i], patchB=nodes[j], shrinkA=2, shrinkB=3,
                                     mutation_scale=max(6 + 6 * T[i, j] / wmax, 11 if hit else 0),
                                     lw=max(.3 + 4.5 * T[i, j] / wmax, 1.8 if hit else 0),   # red never too thin to see
                                     color=HIGHLIGHT if hit else base, alpha=1 if hit else .7,
                                     zorder=3 if hit else 2))
        if hit and note_T is not None:                                  # beside the middle of the curve
            (x1, y1), (x2, y2) = pos[i], pos[j]
            # centred just outside the curve's middle (the curve itself bulges .09 out): beside the
            # line on a long arrow, over it only where a short one leaves no room
            perp = np.array([y2 - y1, x1 - x2])
            mid = (pos[i] + pos[j]) / 2 + .17 * perp
            if not clear_of(mid, note_T[i, j], pos, radii, 1):          # never inside a circle: slide away
                curve = (pos[i] + pos[j]) / 2 + .09 * perp              # from the curve, outward first
                n = perp / np.linalg.norm(perp)
                mid = next((curve + side * t * n for t in (.08, .12, .16, .2, .25, .3, .36, .42)
                            for side in (1, -1) if clear_of(curve + side * t * n, note_T[i, j], pos, radii, 1)), mid)
            ax.text(*mid, note_T[i, j], ha="center", va="center", fontsize=6.5, color=HIGHLIGHT, zorder=5,
                    bbox=dict(boxstyle="round,pad=.15", fc=SURFACE, ec="none", alpha=.85))
    for k in range(K):                                                  # loops and labels
        u, r = pos[k] / np.linalg.norm(pos[k]), radius(occ[k])
        th = np.degrees(np.arctan2(u[1], u[0]))
        ax.add_patch(Arc(pos[k] + u * (r + .045), .11, .11, theta1=th - 140, theta2=th + 140, zorder=2,
                         lw=.3 + 3 * T[k, k], color=HIGHLIGHT if hit_T[k, k] else base))
        occ_note = note_occ[k] if hit_occ[k] and note_occ is not None else None
        ax.text(*pos[k], str(k), ha="center", va="center", fontsize=7.5, color=INK, zorder=6)
        ax.text(*(pos[k] * (1 + (r + .36) / np.linalg.norm(pos[k]))),             # time in state, beside it
                pct(occ[k]) + (f" ({occ_note})" if occ_note else ""), ha="center", va="center",
                fontsize=6.5, color=HIGHLIGHT if occ_note else INK2, zorder=6)
        if hit_T[k, k] and note_T is not None:                          # a red loop's change: from its outer
            side = np.array([u[1], -u[0]])                              # corner, sideways, a little over it
            d = (u + side) / np.linalg.norm(u + side)
            ha, va = away(d)
            ax.annotate(note_T[k, k], pos[k] + u * (r + .045) + d * .055, xytext=-1.5 * d,
                        textcoords="offset points", ha=ha, va=va, fontsize=6.2, color=HIGHLIGHT, zorder=7,
                        bbox=dict(boxstyle="round,pad=.12", fc=SURFACE, ec="none", alpha=.8))
    ax.set_xlim(-1.6, 1.6)
    ax.set_ylim(-1.75, 1.75)                                            # room for the top / bottom labels
    ax.set_aspect("equal")
    ax.axis("off")

def six_diagrams(dyn, hits, title, name, meaning):
    """Rows = tasks; columns = Normal, then each phenotype. dyn[task][group] = (T, occ, n);
    hits[task][group] = (hit_T, hit_occ, note_T, note_occ) for the phenotype columns.
    Arrows: the Normal group's EDGE_TOP strongest plus every highlighted one, on one width scale per row."""
    cols = ["Normal", *TARGETS]
    fig, axes = plt.subplots(len(dyn), len(cols), squeeze=False, figsize=(4.6 * len(cols), 4.7 * len(dyn)))
    for i, (task, D) in enumerate(dyn.items()):
        K = len(D["Normal"][1])
        off = [(a, b) for a in range(K) for b in range(K) if a != b]
        edges = set(sorted(off, key=lambda e: -D["Normal"][0][e])[:EDGE_TOP])
        edges |= {e for g in TARGETS for e in off if hits[task][g][0][e]}
        wmax = max(D[g][0][e] for g in cols for e in edges)
        omax = max(D[g][1].max() for g in cols)
        for j, g in enumerate(cols):
            T, occ, n = D[g]
            if g == "Normal":
                diagram(axes[i, j], T, occ, sorted(edges), wmax, omax)
                axes[i, j].set_title(f"{task} · Normal on both scales (n={n})", fontsize=9.5)
                continue
            hT, hO, nT, nO = hits[task][g]
            diagram(axes[i, j], T, occ, sorted(edges), wmax, omax, ref=D["Normal"][:2],
                    hit_T=hT, hit_occ=hO, note_T=nT, note_occ=nO)
            axes[i, j].set_title(f"{task} · {g}, {CASE_FROM}+ (n={n})  ·  {int(hT.sum() + hO.sum())} in red",
                                 fontsize=9.5)
    fig.legend(handles=[
        Line2D([], [], ls="none", marker="o", ms=10, mfc=NODE, mec=INK2, label="circle area = time in state (%)"),
        Line2D([], [], color=INK2, lw=2.5, label="arrow width = p(j | i): share of moves out of i that go to j"),
        Line2D([], [], ls="none", marker=r"$\circlearrowleft$", ms=12, color=INK2, label="loop = p(stay): thicker = held longer"),
        Line2D([], [], color=HIGHLIGHT, lw=2.5, label=meaning)],
        loc="lower center", ncol=3, fontsize=7.5, bbox_to_anchor=(.5, -.005))
    fig.suptitle(title, x=.01, ha="left", fontsize=10)
    fig.tight_layout(rect=(0, .07, 1, .97))
    save(fig, name)
    plt.show()


def best_set():
    """SET_4, or the state set with the best MODEL / ARM score averaged over the four panels."""
    if SET_4:
        return dict(SET_4, window_s=float(SET_4["window_s"]))
    b = summ[(summ["tree"] == "bstm") & (summ["model"] == MODEL) & (summ["arm"] == ARM["bstm"])]
    m = b.groupby(["atlas", "window_s", "states"])["mean"].agg(["mean", "size"])
    m = m[m["size"] == len(TASKS) * len(TARGETS)]
    return dict(zip(["atlas", "window_s", "states"], m["mean"].idxmax()))

def difference_test(d, K, member, g):
    """The volcano's test on the graph: every p(j | i), p(stay) and occupancy, group g vs Normal."""
    rows = member[g] | member["Normal"]
    dd = d[rows].reset_index(drop=True)
    ind = member[g][rows].astype(float)
    frames = []
    for block, A in {"cells": as_p_cond(dd, K), "occupancy": dd[[f"occ_{k}" for k in range(K)]]}.items():
        A = A.loc[:, A.std() > 0]
        A = A.loc[:, (A > 0).mean() >= MIN_PRESENT]
        x, p = effects(A.to_numpy(float), None, dd[COVARIATES], SHRINK * A.to_numpy(float).mean(), group=ind)
        frames.append(pd.DataFrame({"block": block, "variable": A.columns, "x": x, "p": p}))
    v = pd.concat(frames, ignore_index=True)
    v["p_fdr"] = stats.false_discovery_control(v["p"], method="bh")
    return v.assign(hit=passes(v))

set4 = best_set()
K4 = int(set4["states"].rsplit("_", 1)[1])
dyn4, hits4, tests4 = {}, {}, []
for task in TASKS:
    d = state_table(set4, task)
    m = groups(d)
    dyn4[task] = {g: (*pooled(d, K4, m[g]), int(m[g].sum())) for g in m}
    T0, o0, _ = dyn4[task]["Normal"]
    hits4[task] = {}
    for g in TARGETS:
        v = difference_test(d, K4, m, g)
        tests4.append(v.assign(task=task, group=g))
        hT, hO = np.zeros((K4, K4), bool), np.zeros(K4, bool)
        for r in v[v["hit"]].itertuples():
            if r.block == "cells":
                a, b = map(int, r.variable.split("->"))
                hT[a, b] = True
            else:
                hO[int(r.variable.split("_")[1])] = True
        T, occ, _ = dyn4[task][g]
        hits4[task][g] = (hT, hO, np.vectorize(lambda x: f"{100 * x:+.0f} pp")(T - T0),
                          [f"{100 * x:+.0f} pp" for x in occ - o0])

six_diagrams(dyn4, hits4, f"4  how the dynamics differ from Normal · {set_label(set4)}",
             "fig4_state_diagrams",
             f"red = differs from Normal: {PASS_TEXT}, adjusted (the volcano's rule); label = change vs Normal in pp")
pd.concat(tests4).query("hit")[["task", "group", "block", "variable", "x", "p", "p_fdr"]].round(4)

### Editing notes

* **Figure 1's rows:** `FIG1_MODELS` / `FIG1_ARMS` narrow it. `MARGIN_SD = 1` asks a row to clear its
  baseline + sd by a whole sd of its own as well. `HIGHLIGHT_MAX` sets how many are named per panel.
* **Baseline:** each row of the runs also carries `n_transitions` (bstm) or the motion columns (fcm)
  on top of its features, so the gap to an Age + Sex line includes what those add. The baselines are
  fitted once, in their own cell, so re-drawing a figure does not refit them.
* **Figures 1.2–1.6:** add `"K"` or `"embedding"` to `FACTORS` (as `"K_"` for K), or drop an entry.
* **Figure 2:** `MODEL` / `ARM` pick its sets. With `ARM["bstm"] = "cells [cond]"`, the volcano tests
  the Markov p(j | i) instead of the stored p_joint. `X_AXIS = "r"` gives the correlation volcano.
* **More or fewer sets:** `TOP_N`. Past ten, colours repeat with a new marker shape.
* **A single panel:** call `plot_volcano(ax, volc[(volc.task == "rest") & (volc.phenotype == "anxiety")])`.
* **Hits for one set:** `volc.query("rank == 1 and phenotype == 'anxiety'").nsmallest(10, "p")`.
* `top_sets(task, phenotype)` shows exactly which rows of `summary.csv` the volcano drew from.
* The ranking score and the volcano answer different questions. A set can rank first because its signal
  is spread across many cells and still show no single cell above the FDR line.